# Expert-CFG + RAG 평가 — VQA-RAD

논문 §3 의 Retrieval-Augmented Generation + Classifier-Free Guidance 재현.

**전제**: `VQA_RAD_expert_cfg_reproduction.ipynb` 의 학습 완료 체크포인트가 Google Drive에 있어야 합니다.

In [11]:
# 1) 저장소 클론
!git clone https://github.com/Minseongkim-dev/Expert-CFG.git /content/Expert-CFG 2>/dev/null || \
    (cd /content/Expert-CFG && git pull)

Already up to date.


In [12]:
from google.colab import drive
drive.mount('/content/drive')

# ── 경로 (베이스라인 노트북과 동일하게 맞추세요) ─────────────────────────
BASE_MODEL_DIR   = './Phi-3.5-vision-instruct'
MED_LORA_HF      = 'ecoxial2007/Phi-3.5V-Med'
MED_LORA_DIR     = './lora_weights/logs_phi35_pubmed_instruct'
OUTPUT_DIR       = '/content/drive/MyDrive/expert_cfg/logs_phi35_vqarad_baseline'
RESULT_JSON_CFG  = '/content/drive/MyDrive/expert_cfg/vqarad_cfg_test.json'
_GDRIVE_JSON     = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
_GDRIVE_IMGDIR   = '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder'
RAG_CACHE_DIR    = '/content/drive/MyDrive/expert_cfg/rag_cache'
VQARAD_DIR       = './data/VQA_RAD'

# ── CFG 하이퍼파라미터 ────────────────────────────────────────────────────
GUIDANCE_SCALE = 1.5   # 논문 §4.3
TOP_K          = 3     # RAG 검색 결과 수
MAX_NEW_TOKENS = 64
NUM_CROPS      = 16

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# 패키지 버전 체크 (baseline 노트북과 동일)
import importlib, subprocess, sys

def _ver(name):
    try: return importlib.metadata.version(name)
    except: return None

for _pkg, _min in [('torchao', (0,16)), ('open_clip_torch', None), ('faiss_gpu', None)]:
    _v = _ver(_pkg)
    if _v is None:
        _install = {'open_clip_torch': 'open_clip_torch', 'faiss_gpu': 'faiss-gpu'}.get(_pkg, _pkg)
        print(f'{_pkg} 없음 → 설치 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _install])
    elif _pkg == 'torchao' and tuple(int(x) for x in _v.split('.')[:2]) < _min:
        print(f'torchao {_v} → 업그레이드 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'])
    else:
        print(f'{_pkg} {_v} OK')

torchao 0.18.0 OK
open_clip_torch 3.3.0 OK
faiss_gpu 1.15.1 OK


In [14]:
import os, sys, json, glob, re, random
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

# repo root
REPO_ROOT = Path('/content/Expert-CFG')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import transformers, peft
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch:', torch.__version__, '| transformers:', transformers.__version__)
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('BF16 :', USE_BF16)

torch: 2.11.0+cu130 | transformers: 5.20.2
GPU  : NVIDIA A100-SXM4-40GB
BF16 : True


In [15]:
# ── 경로 ────────────────────────────────────────────────────────────────
BASE_MODEL_DIR = './Phi-3.5-vision-instruct'
MED_LORA_DIR   = './lora_weights/logs_phi35_pubmed_instruct'
MED_LORA_HF    = 'ecoxial2007/Phi-3.5V-Med'
VQARAD_DIR     = './data/VQA_RAD'

# ── LoRA (논문 §4.3) ─────────────────────────────────────────────────────
LORA_RANK        = 64
LORA_ALPHA_RATIO = 2
LORA_DROPOUT     = 0.0
LORA_TARGET      = ['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj']
FREEZE_VISION    = True

# ── 학습 (§4.3) ──────────────────────────────────────────────────────────
LR            = 1e-4
GLOBAL_BATCH  = 64
EPOCHS        = 5
PER_DEVICE_BS = 1
WORLD_SIZE    = int(os.environ.get('WORLD_SIZE', 1))
GRAD_ACCUM    = GLOBAL_BATCH // (PER_DEVICE_BS * WORLD_SIZE)
WARMUP_RATIO  = 0.03
WEIGHT_DECAY  = 0.0

# ── 추론/학습 이미지 크롭 ─────────────────────────────────────────────────
# 논문은 16크롭이지만 T4(16GB)에서는 OOM → 4로 줄임
# A100 사용 시 16으로 변경 가능
_gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ''
NUM_CROPS = 16
print(f'NUM_CROPS={NUM_CROPS}  (GPU: {_gpu_name or "cpu"})')

# T4는 BF16 네이티브 미지원 → FP16 사용
USE_BF16 = torch.cuda.is_available() and 'a100' in _gpu_name
USE_FP16 = torch.cuda.is_available() and not USE_BF16
print(f'precision: {"bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32"}')

# ── 추론 ─────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 64
NUM_BEAMS      = 1
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULT_JSON_CFG), exist_ok=True)
print(f'grad_accum={GRAD_ACCUM}  (global batch {GLOBAL_BATCH})')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

NUM_CROPS=16  (GPU: nvidia a100-sxm4-40gb)
precision: bf16
grad_accum=64  (global batch 64)
OUTPUT_DIR = /content/drive/MyDrive/expert_cfg/logs_phi35_vqarad_baseline


In [16]:
import shutil
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from peft import PeftModel, LoraConfig, get_peft_model

_SKIP_FILES = {'processor_config.json', 'processing_phi3_v_cfg.py'}

# 1) 기본 가중치 다운로드
if not os.path.exists(os.path.join(BASE_MODEL_DIR, 'config.json')):
    print('Phi-3.5-vision-instruct 가중치 다운로드 중 (~8 GB)...')
    _hf_tmp = snapshot_download('microsoft/Phi-3.5-vision-instruct',
                                local_dir='/tmp/phi35_hf',
                                ignore_patterns=['*.git', '__pycache__'])
    os.makedirs(BASE_MODEL_DIR, exist_ok=True)
    for _f in os.listdir(_hf_tmp):
        if _f in _SKIP_FILES:
            continue
        _s, _d = os.path.join(_hf_tmp, _f), os.path.join(BASE_MODEL_DIR, _f)
        (shutil.copytree if os.path.isdir(_s) else shutil.copy2)(
            _s, _d, **({'dirs_exist_ok': True} if os.path.isdir(_s) else {}))
    print('완료')
else:
    print('기본 가중치 이미 존재')

with open(os.path.join(BASE_MODEL_DIR, 'processor_config.json')) as _f:
    assert 'processing_phi3_v_cfg' in _f.read(), \
        'processor_config.json 이 HF 원본으로 덮어써졌습니다 — repo 버전으로 복원하세요'

processor = AutoProcessor.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

# 2) Med LoRA 다운로드/확인
def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    assert found, f'{MED_LORA_HF} 에서 adapter_config.json 을 찾지 못했습니다'
    return os.path.dirname(found[0])

med_lora_path = resolve_med_lora()
print('Med LoRA :', med_lora_path)

# 3) Base → Med LoRA merge → 새 downstream LoRA 추가 (fine-tuning 준비)
_model_dtype = torch.bfloat16 if USE_BF16 else torch.float16
print(f'모델 로드 dtype: {_model_dtype}')

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR, dtype=_model_dtype,
    trust_remote_code=True, _attn_implementation='eager')

model = PeftModel.from_pretrained(base, med_lora_path).merge_and_unload()

model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_RANK * LORA_ALPHA_RATIO,
    lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET,
    init_lora_weights=True, task_type='CAUSAL_LM'))

if FREEZE_VISION:
    for n, p in model.named_parameters():
        if 'vision_embed_tokens' in n:
            p.requires_grad_(False)

# transformers 5.x + PEFT: merge_and_unload() 후 generation_config 가 유실되는 버그 대응
try:
    _gen_cfg = GenerationConfig.from_pretrained(BASE_MODEL_DIR)
except Exception:
    _gen_cfg = GenerationConfig()
model.generation_config = _gen_cfg

model.print_trainable_parameters()
model.to(DEVICE)
print('모델 준비 완료 (fine-tuning 준비 상태)')

[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'


기본 가중치 이미 존재


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'
[transformers] Phi3VForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Med LoRA : /content/Expert-CFG/lora_weights/Phi-3.5V-Med
모델 로드 dtype: torch.bfloat16


Loading weights:   0%|          | 0/592 [00:00<?, ?it/s]

[transformers] Phi3VForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


trainable params: 100,663,296 || all params: 4,247,284,736 || trainable%: 2.3701
모델 준비 완료 (fine-tuning 준비 상태)


In [ ]:
import subprocess
from PIL import Image
from datasets import load_dataset

# ── Google Drive 업로드 경로 ───────────────────────────────────────────
_GDRIVE_JSON   = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
_GDRIVE_IMGDIR = '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder'

def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

def _prepare_local():
    train_json = os.path.join(VQARAD_DIR, 'train.json')
    test_json  = os.path.join(VQARAD_DIR, 'test.json')
    if os.path.exists(train_json) and os.path.exists(test_json):
        print('로컬 train/test JSON 이미 존재 → 재사용')
        return True

    raw_candidates = [
        _GDRIVE_JSON,
        os.path.join(VQARAD_DIR, 'VQA_RAD Dataset Public.json'),
        os.path.join(VQARAD_DIR, 'VQA_RAD_Dataset_Public.json'),
    ]
    raw_json = None
    for _p in raw_candidates:
        if os.path.exists(_p):
            _sz = os.path.getsize(_p)
            if _sz > 0:
                raw_json = _p
                print(f'원본 JSON 발견: {_p}  ({_sz:,} bytes)')
                break
            else:
                print(f'파일 비어있음(0 bytes): {_p}')

    if raw_json is None:
        os.makedirs(VQARAD_DIR, exist_ok=True)
        _osf_dst = os.path.join(VQARAD_DIR, 'VQA_RAD_Dataset_Public.json')
        print('OSF에서 VQA-RAD 원본 다운로드 중...')
        try:
            r = subprocess.run(
                ['wget', '-q', '-O', _osf_dst, 'https://osf.io/download/9dpze/'],
                timeout=120, capture_output=True)
            if r.returncode == 0 and os.path.getsize(_osf_dst) > 0:
                raw_json = _osf_dst
            else:
                return False
        except Exception as e:
            print(f'  OSF 다운로드 실패: {e}')
            return False

    with open(raw_json) as f:
        all_data = json.load(f)

    # answer_type 정규화
    for item in all_data:
        if 'answer_type' not in item:
            item['answer_type'] = answer_type_of(item.get('answer', ''))
        else:
            item['answer_type'] = item['answer_type'].upper()

    # ── phrase_type 기반 split ──────────────────────────────────────
    # README §5: test = test_freeform(300) + test_paraphrase(151) = 451
    #            train = freeform + para
    _TEST_TYPES = {'test_freeform', 'test_para'}
    _phrase_vals = {str(r.get('phrase_type', '')).lower() for r in all_data}
    print(f'phrase_type 고유값: {_phrase_vals}')

    if _TEST_TYPES & _phrase_vals:
        test_data  = [r for r in all_data if str(r.get('phrase_type','')).lower() in _TEST_TYPES]
        train_data = [r for r in all_data if str(r.get('phrase_type','')).lower() not in _TEST_TYPES]
    elif 'para' in _phrase_vals:
        # 구버전 JSON: freeform/para 만 있을 때 — para 를 test 로 쓰면 안 되므로
        # 뒤 451 개를 test 로 fallback
        print('경고: test_freeform/test_paraphrase 없음 → 뒤 451개를 test로 사용')
        test_data  = all_data[-451:]
        train_data = all_data[:-451]
    else:
        test_data  = all_data[-451:]
        train_data = all_data[:-451]

    os.makedirs(VQARAD_DIR, exist_ok=True)
    with open(train_json, 'w') as f: json.dump(train_data, f)
    with open(test_json,  'w') as f: json.dump(test_data,  f)
    print(f'분리 완료: train={len(train_data)}, test={len(test_data)}')
    print(f'  test closed={sum(1 for r in test_data if r["answer_type"]=="CLOSED")}, '
          f'open={sum(1 for r in test_data if r["answer_type"]=="OPEN")}')
    return True

def _find_img_root():
    for p in [_GDRIVE_IMGDIR,
              os.path.join(VQARAD_DIR, 'VQA_RAD Image Folder'),
              os.path.join(VQARAD_DIR, 'images')]:
        if os.path.isdir(p):
            return p
    return os.path.join(VQARAD_DIR, 'images')

_HF_CANDIDATES = {
    'train': [('flaviagiammarino/vqa-rad', 'train')],
    'test':  [('flaviagiammarino/vqa-rad', 'test')],
}

def _try_load_hf(split, cache_dir):
    for ds_id, sp in _HF_CANDIDATES[split]:
        try:
            ds = load_dataset(ds_id, split=sp, cache_dir=cache_dir)
            print(f'  HF 로드: {ds_id}/{sp}  ({len(ds)})')
            return ds, ds_id
        except Exception as e:
            print(f'  {ds_id}/{sp} 실패: {type(e).__name__}')
    raise RuntimeError('VQA-RAD 데이터를 가져오지 못했습니다.')

class VQARADSplit(torch.utils.data.Dataset):
    def __init__(self, split):
        local_json = os.path.join(VQARAD_DIR, f'{split}.json')
        if os.path.exists(local_json):
            with open(local_json) as f:
                self.records = json.load(f)
            self.source   = 'local'
            self.img_root = _find_img_root()
            self.ds_id    = 'local'
            print(f'[{split}] 로컬 JSON ({len(self.records)}개), 이미지: {self.img_root}')
        else:
            print(f'[{split}] HuggingFace 로드...')
            hf_ds, self.ds_id = _try_load_hf(split, os.path.join(VQARAD_DIR, 'hf_cache'))
            self.records = hf_ds
            self.source  = 'hf'

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        if self.source == 'local':
            img_name = (r.get('image_name') or r.get('img_name')
                        or r.get('image') or r.get('img_id', ''))
            image = Image.open(os.path.join(self.img_root, img_name)).convert('RGB')
            q  = str(r.get('question', '')).strip()
            a  = str(r.get('answer',   '')).strip()
            at = str(r.get('answer_type', '')).upper()
            if at not in ('OPEN', 'CLOSED'):
                at = answer_type_of(a)
        else:
            q   = str(r.get('question') or '').strip()
            a   = str(r.get('answer')   or '').strip()
            img = r.get('image') or r.get('img')
            image = img.convert('RGB') if img else Image.new('RGB', (224, 224))
            at  = str(r.get('answer_type', '')).upper()
            if at not in ('OPEN', 'CLOSED'):
                at = answer_type_of(a)
        return {'image': image, 'question': q, 'answer': a, 'answer_type': at}

def type_counts(ds):
    c = {'OPEN': 0, 'CLOSED': 0}
    for i in range(len(ds)):
        c[ds[i]['answer_type']] += 1
    return c['CLOSED'], c['OPEN']

# ── 실행 ─────────────────────────────────────────────────────────────
_ok = _prepare_local()
if not _ok:
    print('경고: 로컬 JSON 준비 실패 → HuggingFace 폴백')

train_ds = VQARADSplit('train')
test_ds  = VQARADSplit('test')

for name, ds in [('train', train_ds), ('test', test_ds)]:
    c, o = type_counts(ds)
    print(f'{name:5s} [{ds.ds_id}] total={len(ds):4d}  closed={c:3d}  open={o:3d}')
print('논문 기준: test total=451')

원본 JSON 발견: /content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json  (1,419,295 bytes)
phrase_type 고유값: {'test_para', 'freeform', 'test_freeform', 'para'}
분리 완료: train=1797, test=451
  test closed=272, open=179
[train] 로컬 JSON (1797개), 이미지: /content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder
[test] 로컬 JSON (451개), 이미지: /content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder


In [6]:
# ── transformers 5.x 호환: Phi3VForCausalLM _attn_implementation 패치 ──
_mfile = os.path.join(BASE_MODEL_DIR, 'modeling_phi3_v.py')
if os.path.exists(_mfile):
    with open(_mfile) as _f:
        _src = _f.read()
    _patched = _src
    for _old, _new in [
        ('def __init__(self, config: Phi3VConfig):',  'def __init__(self, config: Phi3VConfig, **kwargs):'),
        ('def __init__(self, config: PretrainedConfig):', 'def __init__(self, config: PretrainedConfig, **kwargs):'),
    ]:
        _patched = _patched.replace(_old, _new)
    if _patched == _src:
        _patched = re.sub(
            r'(def __init__\(self, config[^)]*?)(\):)',
            lambda m: m.group(0) if '**kwargs' in m.group(0) else m.group(1) + ', **kwargs' + m.group(2),
            _src
        )
    if _patched != _src:
        with open(_mfile, 'w') as _f:
            _f.write(_patched)
        print('modeling_phi3_v.py 패치 완료')
    else:
        print('이미 패치됨 / 패턴 미발견')
else:
    print(f'파일 없음: {_mfile}  (모델 다운로드 셀 먼저 실행)')

파일 없음: ./Phi-3.5-vision-instruct/modeling_phi3_v.py  (모델 다운로드 셀 먼저 실행)


In [7]:
# ── VQA-RAD 데이터 로드 ───────────────────────────────────────────────────
def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

def load_split(split):
    local_json = os.path.join(VQARAD_DIR, f'{split}.json')
    if os.path.exists(local_json):
        with open(local_json) as f:
            return json.load(f)
    raise FileNotFoundError(
        f'{local_json} 없음 — 베이스라인 노트북을 먼저 실행해 데이터를 준비하세요.')

def _find_img_root():
    for p in [_GDRIVE_IMGDIR,
              os.path.join(VQARAD_DIR, 'VQA_RAD Image Folder'),
              os.path.join(VQARAD_DIR, 'images')]:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError('이미지 폴더를 찾지 못했습니다.')

train_anns = load_split('train')
test_anns  = load_split('test')
IMG_ROOT   = _find_img_root()

# 베이스라인 JSON 은 image_name 키를 쓰므로 RAG 가 기대하는 img_id 로 통일
for _r in train_anns + test_anns:
    if 'img_id' not in _r:
        _r['img_id'] = _r.get('image_name') or _r.get('img_name') or _r.get('image', '')

# answer_type 정규화
for _r in train_anns + test_anns:
    if 'answer_type' not in _r:
        _r['answer_type'] = answer_type_of(_r.get('answer', ''))
    else:
        _r['answer_type'] = _r['answer_type'].upper()

print(f'train={len(train_anns)}, test={len(test_anns)}')
print(f'이미지 경로: {IMG_ROOT}')

FileNotFoundError: ./data/VQA_RAD/train.json 없음 — 베이스라인 노트북을 먼저 실행해 데이터를 준비하세요.

In [8]:
# ── RAG: BioMedCLIP 로드 + FAISS 인덱스 구축 ────────────────────────────
from src.rag.biomedclip_rag import BioMedCLIPRAG

rag = BioMedCLIPRAG(device=DEVICE)
rag.load_model()

# training set 이미지 피처 추출 및 인덱스 빌드
# cache_dir 에 저장하면 다음 실행부터 즉시 로드됨
rag.build_index(train_anns, IMG_ROOT, cache_dir=RAG_CACHE_DIR)

# 검색 테스트
_test_img = os.path.join(IMG_ROOT, test_anns[0]['img_id'])
_test_results = rag.retrieve(_test_img, top_k=TOP_K)
print('\n검색 예시 (test[0]):')
print('  Q:', test_anns[0]['question'])
for _r in _test_results:
    print(f'  → {_r["question"]}  /  {_r["answer"]}')

ModuleNotFoundError: No module named 'src.rag'

In [ ]:
# ── Phi-3.5V + Med LoRA + VQA-RAD LoRA 로드 ─────────────────────────────
import glob as _glob
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from peft import PeftModel

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

# Med LoRA 경로 찾기
def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    from huggingface_hub import snapshot_download
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = _glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    return os.path.dirname(found[0])

_dtype = torch.bfloat16 if USE_BF16 else torch.float16

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR, dtype=_dtype,
    trust_remote_code=True, _attn_implementation='eager')

# Med LoRA merge
model = PeftModel.from_pretrained(base, resolve_med_lora()).merge_and_unload()

# VQA-RAD LoRA 로드
model = PeftModel.from_pretrained(model, OUTPUT_DIR).to(DEVICE).eval()

try:
    model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_DIR)
except Exception:
    model.generation_config = GenerationConfig()

print('모델 로드 완료')

In [ ]:
# ── DynamicCache 패치 (transformers 5.x seen_tokens 복원) ────────────────
import importlib

_patched = []
for _mod_path in ['transformers.cache_utils', 'transformers.generation.cache_utils']:
    try:
        _mod = importlib.import_module(_mod_path)
        _DC  = getattr(_mod, 'DynamicCache', None)
        if _DC is None:
            continue
        _DC.seen_tokens = property(lambda self: self.get_seq_length())
        def _get_usable_length(self, new_seq_length, layer_idx=0):
            return self.get_seq_length(layer_idx)
        _DC.get_usable_length = _get_usable_length
        _DC.get_max_length    = lambda self: None
        _patched.append(_mod_path)
    except Exception as _e:
        print(f'  {_mod_path} 패치 실패: {_e}')

print('DynamicCache 패치:', _patched)

# 검증
from transformers.cache_utils import DynamicCache as _DC2
_dc = _DC2(); print(f'seen_tokens = {_dc.seen_tokens}  (0 이면 정상)'); del _dc

In [ ]:
# ── CFG 생성 함수 ─────────────────────────────────────────────────────────
import sys
from transformers import LogitsProcessorList
from src.highlighter_modules.guidance import ProbCFGLogitsProcessor

def _left_pad(ids, target_len, pad_id):
    """ids: (1, L) → (1, target_len) with left-padding."""
    L = ids.shape[1]
    if L == target_len:
        return ids, torch.ones_like(ids)
    pad = torch.full((1, target_len - L), pad_id, dtype=ids.dtype)
    return (torch.cat([pad, ids], dim=1),
            torch.cat([torch.zeros(1, target_len - L, dtype=torch.long), torch.ones_like(ids)], dim=1))


def prepare_cfg_batch(processor, image, question, context, device):
    """
    Returns doubled-batch inputs for CFG:
      row 0: conditional   (image + context + question)
      row 1: unconditional (image + question only)
    """
    cond_text = processor.tokenizer.apply_chat_template(
        [{'role': 'user',
          'content': f'<|image_1|>\nContext:\n{context}\n\nQuestion: {question}'}],
        tokenize=False, add_generation_prompt=True
    )
    uncond_text = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': f'<|image_1|>\n{question}'}],
        tokenize=False, add_generation_prompt=True
    )

    cond_in  = processor(cond_text,  [image], return_tensors='pt', qs_highlighted_parts=[])
    uncond_in = processor(uncond_text, [image], return_tensors='pt', qs_highlighted_parts=[])
    for _d in [cond_in, uncond_in]:
        _d.pop('highlight_attention_mask', None)

    pad_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id
    L = max(cond_in['input_ids'].shape[1], uncond_in['input_ids'].shape[1])

    c_ids,  c_mask  = _left_pad(cond_in['input_ids'],   L, pad_id)
    u_ids,  u_mask  = _left_pad(uncond_in['input_ids'], L, pad_id)

    batch = {
        'input_ids':      torch.cat([c_ids,  u_ids],  dim=0).to(device),
        'attention_mask': torch.cat([c_mask, u_mask], dim=0).to(device),
    }

    # pixel_values: duplicate same image for both batch items
    pv = cond_in.get('pixel_values')
    if pv is not None:
        batch['pixel_values'] = torch.cat([pv, pv], dim=0).to(device)

    img_sz = cond_in.get('image_sizes')
    if img_sz is not None:
        batch['image_sizes'] = torch.cat([img_sz, img_sz], dim=0).to(device)

    return batch


@torch.no_grad()
def generate_cfg(model, processor, image, question, context,
                 guidance_scale=1.5, max_new_tokens=64, device='cuda'):
    """
    Run CFG decoding:
      guided_logits = scale*(cond - uncond) + uncond
    Returns decoded string.
    """
    model.eval()
    end_id = processor.tokenizer.convert_tokens_to_ids('<|end|>')

    batch = prepare_cfg_batch(processor, image, question, context, device)
    cfg_proc = ProbCFGLogitsProcessor(guidance_scale=guidance_scale)

    out = model.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        eos_token_id=[processor.tokenizer.eos_token_id, end_id],
        logits_processor=LogitsProcessorList([cfg_proc]),
        use_cache=True,
        output_scores=False,
        return_dict_in_generate=False,
    )

    # out: (2, full_len) — both rows have same generated tokens; take row 0
    prefix_len = batch['input_ids'].shape[1]
    new_ids = out[0, prefix_len:]
    return processor.tokenizer.decode(new_ids, skip_special_tokens=True).strip().strip('.')


print('CFG 생성 함수 정의 완료')

In [ ]:
# ── 평가 지표 ────────────────────────────────────────────────────────────
def normalize(text):
    text = text.lower().replace('<|end|>', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(gt.strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def summarize(records):
    closed = [r['score'] for r in records if r['answer_type'] == 'CLOSED']
    opened = [r['score'] for r in records if r['answer_type'] == 'OPEN']
    return {
        'open':    100 * np.mean(opened) if opened else float('nan'),
        'closed':  100 * np.mean(closed) if closed else float('nan'),
        'overall': 100 * np.mean([r['score'] for r in records]),
        'n_open': len(opened), 'n_closed': len(closed),
    }

print('평가 함수 정의 완료')

In [ ]:
# ── CFG 평가 실행 ────────────────────────────────────────────────────────
records_cfg = []

for ann in tqdm(test_anns, desc=f'CFG eval (scale={GUIDANCE_SCALE})'):
    img_path = os.path.join(IMG_ROOT, ann.get('img_id', ann.get('image_name', '')))
    image    = Image.open(img_path).convert('RGB')
    question = ann['question']
    gt       = ann['answer']
    atype    = ann.get('answer_type', answer_type_of(gt))

    # RAG 검색
    retrieved = rag.retrieve(img_path, top_k=TOP_K)
    context   = BioMedCLIPRAG.format_context(retrieved)

    # CFG 생성
    pred = generate_cfg(
        model, processor, image, question, context,
        guidance_scale=GUIDANCE_SCALE,
        max_new_tokens=MAX_NEW_TOKENS,
        device=DEVICE,
    )

    score = (closed_accuracy if atype == 'CLOSED' else open_recall)(pred, gt)
    records_cfg.append({
        'question': question, 'answer': gt, 'answer_type': atype,
        'prediction': pred, 'score': score, 'context': context,
    })

with open(RESULT_JSON_CFG, 'w') as f:
    json.dump(records_cfg, f, indent=2, ensure_ascii=False)

m_cfg = summarize(records_cfg)
paper = {'open': 69.27, 'closed': 90.44, 'overall': 82.04}

print(f"{'':8s}{'paper':>8s}{'CFG':>8s}{'diff':>8s}")
for k in ('open', 'closed', 'overall'):
    print(f'{k:8s}{paper[k]:8.2f}{m_cfg[k]:8.2f}{m_cfg[k]-paper[k]:+8.2f}')
print(f"n_open={m_cfg['n_open']}, n_closed={m_cfg['n_closed']}")
print(f'결과: {RESULT_JSON_CFG}')

In [ ]:
# ── guidance_scale 스윕 (빠른 검증용 — 50개 서브셋) ──────────────────────
import random
random.seed(42)
_subset = random.sample(test_anns, 50)

for _scale in [1.0, 1.2, 1.5, 2.0, 3.0]:
    _recs = []
    for ann in tqdm(_subset, desc=f'scale={_scale}', leave=False):
        img_path = os.path.join(IMG_ROOT, ann.get('img_id', ann.get('image_name', '')))
        image    = Image.open(img_path).convert('RGB')
        retrieved = rag.retrieve(img_path, top_k=TOP_K)
        context   = BioMedCLIPRAG.format_context(retrieved)
        pred = generate_cfg(
            model, processor, image, ann['question'], context,
            guidance_scale=_scale, max_new_tokens=MAX_NEW_TOKENS, device=DEVICE,
        )
        atype = ann.get('answer_type', answer_type_of(ann['answer']))
        score = (closed_accuracy if atype == 'CLOSED' else open_recall)(pred, ann['answer'])
        _recs.append({'score': score, 'answer_type': atype})
    _m = summarize(_recs)
    print(f'scale={_scale:.1f}  open={_m["open"]:5.1f}  closed={_m["closed"]:5.1f}  overall={_m["overall"]:5.1f}')

## 결과 해석

- `guidance_scale=1.0` → CFG 없음 (conditional 만 사용) ≒ 베이스라인
- `guidance_scale=1.5` → 논문 기본 설정
- scale이 너무 높으면 오히려 성능 저하 (분포 왜곡)

VQA-RAD 전체 CFG 평가는 약 30-60분 소요 (A100 기준).